# Thêm Thư Viện

In [7]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [8]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2024;'
)

## Đọc data từ SQL Server

In [9]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Kho= """SELECT Kho_ID, dbo.DecodeUTF8String(Kho) AS Kho, Thu_vien_ID, MaxID, Mo FROM Kho """
df_kho = pd.read_sql(query_Kho, conn_libol)

query_Thuvien = """SELECT Thu_vien_ID, dbo.DecodeUTF8String(Ten_viet_tat) AS Ten_viet_tat FROM Thu_vien """
df_thuvien = pd.read_sql(query_Thuvien, conn_libol)

print(df_kho)
print(df_thuvien)

   Kho_ID           Kho  Thu_vien_ID  MaxID     Mo
0       7            KM            2      2   True
1       8            NV            1  71094   True
2       9            GT            1  60109   True
3       5            KM            1  14805   True
4       6            KD            1  10574  False
5      19   Đọc tại chỗ            1      1  False
6      13  Kho thanh lý            1      1  False
7      14   Unavailbale            1      1  False
8      16           CLC            1    104  False
9      18       Kho Lưu            1      4  False
    Thu_vien_ID                  Ten_viet_tat
0             1                        DHSPKT
1             2                        ĐHSPKT
2             3                       SDHSPKT
3             4                   ĐHSPKT##Vie
4             5                           Vie
5             6                   DHSPKT##Vie
6             7                   D9HSP T.HCM
7             8                        SPDHKT
8             9          

C:\Users\admin\AppData\Local\Temp\ipykernel_15292\2520694761.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_kho = pd.read_sql(query_Kho, conn_libol)
C:\Users\admin\AppData\Local\Temp\ipykernel_15292\2520694761.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_thuvien = pd.read_sql(query_Thuvien, conn_libol)


## Xử lý data

In [10]:
map_dict = dict(zip(df_thuvien['Thu_vien_ID'], df_thuvien['Ten_viet_tat']))

for index, row in df_kho.iterrows():
    id_thu_vien = row['Thu_vien_ID']
    if id_thu_vien in map_dict:
        # Thay thế ID_Thu_vien bằng Ten_viet_tat nếu khớp
        df_kho.at[index, 'Thu_vien_ID'] = map_dict[id_thu_vien]

new_row = pd.DataFrame({'Kho_ID': [0], # Tạo hàng dữ liệu giả lập cho kho không xác định
                        'Kho': ['(Không xác định)'],
                        'Thu_vien_ID': ['0'],
                        'MaxID': [0], 
                        'Mo': ['False']})
df_kho = pd.concat([df_kho, new_row], ignore_index=True) # Thêm vào dataframe

df_kho = df_kho.sort_values(by="Kho_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_kho)

    Kho_ID               Kho Thu_vien_ID  MaxID     Mo
0        0  (Không xác định)           0      0  False
1        5                KM      DHSPKT  14805   True
2        6                KD      DHSPKT  10574  False
3        7                KM      ĐHSPKT      2   True
4        8                NV      DHSPKT  71094   True
5        9                GT      DHSPKT  60109   True
6       13      Kho thanh lý      DHSPKT      1  False
7       14       Unavailbale      DHSPKT      1  False
8       16               CLC      DHSPKT    104  False
9       18           Kho Lưu      DHSPKT      4  False
10      19       Đọc tại chỗ      DHSPKT      1  False


C:\Users\admin\AppData\Local\Temp\ipykernel_15292\1537312163.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ĐHSPKT' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_kho.at[index, 'Thu_vien_ID'] = map_dict[id_thu_vien]


## Load data

### [Nếu cần] Clear bảng

In [11]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM DIM_Kho"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [12]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO DIM_Kho (ID_kho, Kho, ID_thu_vien, MaxID, Mo) 
                VALUES (?, ?, ?, ?, ?)
               """
for index, row in df_kho.iterrows():
    values = (row['Kho_ID'], 
                row['Kho'],
                row['Thu_vien_ID'],
                row['MaxID'],
                row['Mo'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()